# C0 budget curve (untrained Qwen) — Kaggle shell
Setup: **Accelerator = GPU T4 x2**, Internet on.
1. Upload `dist/brickexp_code.zip` (from `python tools/make_kaggle_bundle.py`) as a **private Kaggle Dataset** and attach it, *or* set `GIT_URL` below.
2. Optional (mechanics checker): add Kaggle **Secrets** `GRB_WLSACCESSID`, `GRB_WLSSECRET`, `GRB_LICENSEID`. Without them the checker falls back to connectivity.
The license file is written to `/tmp` (not `/kaggle/working`, which is saved as output) and is never printed.

In [ ]:
%%capture pipout
!pip install -q -U vllm gurobipy datasets networkx

In [ ]:
import os, sys, glob, zipfile, subprocess
GIT_URL = ""  # optional: a git repo containing brickexp/ and vendor/BrickGPT
WORK = "/kaggle/working/RLtheLEGO"
zips = glob.glob("/kaggle/input/**/brickexp_code.zip", recursive=True)
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/brickexp/__init__.py", recursive=True)]
if not os.path.exists(WORK + "/brickexp"):
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    elif dirs:
        subprocess.run(["cp", "-r", dirs[0], WORK], check=True)
    elif GIT_URL:
        subprocess.run(["git", "clone", "--recursive", GIT_URL, WORK], check=True)
    else:
        raise RuntimeError("Attach the brickexp_code dataset or set GIT_URL")
os.chdir(WORK); sys.path.insert(0, WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Gurobi WLS license from Kaggle Secrets (values never printed)
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    vals = {k: sc.get_secret(k) for k in ("GRB_WLSACCESSID", "GRB_WLSSECRET", "GRB_LICENSEID")}
    with open("/tmp/gurobi.lic", "w") as f:
        f.write("".join(f"{k[4:]}={v}\n" for k, v in vals.items()))
    os.environ["GRB_LICENSE_FILE"] = "/tmp/gurobi.lic"
    del vals
    print("Gurobi secrets loaded")
except Exception as e:
    print("No Gurobi secrets -> connectivity fallback:", type(e).__name__)
from brickexp import checker
print("gurobi available:", checker.gurobi_available())

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3

In [ ]:
# ---- config ----
MODEL = "Qwen/Qwen3.5-2B"      # fallback if unsupported: "Qwen/Qwen3.5-0.8B"
BACKEND = "vllm"               # "vllm" or "hf" (transformers fallback)
PER_BUCKET = 10                 # prompts per size bucket (small/medium/large)
MAX_BUDGET = 10                # retries; one run gives success@k for all k <= MAX_BUDGET (use 50 for the full curve)
HISTORY_WINDOW = 2             # failed attempts kept in context
CHECKER = "connectivity"       # "auto" = Gurobi if licensed (~10-30 s/check on CPU!), else connectivity
TEMPERATURE = 0.6
OUT = f"/kaggle/working/runs/c0_{MODEL.split('/')[-1]}_b{MAX_BUDGET}_{CHECKER}.jsonl"


In [ ]:
from brickexp.data import get_eval_set
recs = get_eval_set(10)  # fixed, seeded subset shipped in data/splits
recs = [r for b in ("small", "medium", "large") for r in [x for x in recs if x["bucket"] == b][:PER_BUCKET]]
print(len(recs), "prompts")

In [ ]:
import time, torch
from brickexp.backends import make_backend
t = time.time()
kw = dict(model=MODEL, temperature=TEMPERATURE, max_tokens=4096)
if BACKEND == "vllm":
    try:
        be = make_backend("vllm", **kw, dtype="half", tensor_parallel_size=1, max_model_len=16384,
                          limit_mm_per_prompt={"image": 0, "video": 0})
    except Exception as e:
        print("vLLM failed, falling back to HF:", repr(e)[:300]); BACKEND = "hf"
if BACKEND == "hf":
    be = make_backend("hf", **kw, batch_size=8)
print(f"model load {time.time()-t:.0f}s")

In [ ]:
import json
from brickexp.evaluate import run_eval, success_curve, failure_breakdown
ug = {"auto": "auto", "gurobi": True, "connectivity": False}[CHECKER]
t = time.time()
states = run_eval(recs, be, max_budget=MAX_BUDGET, use_gurobi=ug, history_window=HISTORY_WINDOW, out_path=OUT)
print(f"total {time.time()-t:.0f}s  -> {OUT}")
print(json.dumps(success_curve(states, budgets=(0, 1, 10, 50)), indent=1))
print("failures:", failure_breakdown(states))

In [ ]:
import pandas as pd
df = pd.DataFrame([{"bucket": s["bucket"], "n_ref": s["n_bricks_ref"], "solved_at": s["solved_at"],
                    "attempts": len(s["attempts"]), "gen_s": round(s["gen_seconds"], 2),
                    "check_s": round(s["check_seconds"], 2), "s_per_attempt": round(s["seconds"] / len(s["attempts"]), 2)}
                   for s in states])
print(df.to_string()); print(df.groupby("bucket")[["gen_s", "check_s", "s_per_attempt"]].mean())

In [ ]:
# Gurobi cost per check on this CPU (only if licensed), measured on model outputs that passed connectivity
if checker.gurobi_available():
    cands = [a["text"] for s in states for a in s["attempts"] if a["ok"]][:3] or [r["bricks"] for r in recs[:3]]
    for txt in cands:
        t = time.time(); r = checker.check(txt, use_gurobi=True); print(r.n_bricks, r.failure, r.method, f"{time.time()-t:.1f}s")